# Exp138 - build the incumbent + v34 model SOUP (CPU, weight-space average)

Pure weight surgery, **no inference, no GPU** (weight averaging is trivial). Writes
one souped `edge_predictor_best.pth` to be packaged as a dataset and run as a
SINGLE model - which is the only ensemble-like form that survives the submission
notebook rerun (multi-model inference times out; see LEARNINGS).

Both checkpoints are the identical `unet_transformer` architecture (config.json
matches), so a per-tensor average is well defined. `v34` is an *independent*
retrain, so the two may not be weight-aligned - if the soup is incoherent it will
show up as a broken submission when we run inference; that is the risk we are
buying cheaply here.

Float tensors are averaged 50/50; integer buffers (`num_batches_tracked`) keep the
incumbent's value (eval-irrelevant).


In [ ]:
import torch, glob, json, os, hashlib
from pathlib import Path

# incumbent: the support pack primary (has "weights/.../edge_predictor_best.pth")
inc = glob.glob("/kaggle/input/**/weights/unet_transformer/split_0/edge_predictor_best.pth", recursive=True)
# v34: independent retrain (no "weights/" prefix in its path)
v34 = [p for p in glob.glob("/kaggle/input/**/*v34*/**/edge_predictor_best.pth", recursive=True)]
assert inc, "incumbent checkpoint not found"
assert v34, "v34 checkpoint not found"
INC, V34 = inc[0], v34[0]
print("incumbent:", INC)
print("v34      :", V34)

sd_i = torch.load(INC, map_location="cpu", weights_only=True)
sd_v = torch.load(V34, map_location="cpu", weights_only=True)
# both should be flat state dicts
if isinstance(sd_i, dict) and "model_state_dict" in sd_i: sd_i = sd_i["model_state_dict"]
if isinstance(sd_v, dict) and "model_state_dict" in sd_v: sd_v = sd_v["model_state_dict"]

ki, kv = set(sd_i), set(sd_v)
assert ki == kv, f"key mismatch: only-incumbent={sorted(ki-kv)[:5]} only-v34={sorted(kv-ki)[:5]}"
for k in ki:
    assert sd_i[k].shape == sd_v[k].shape, f"shape mismatch at {k}: {sd_i[k].shape} vs {sd_v[k].shape}"
print(f"compatible: {len(ki)} tensors, all shapes match")


In [ ]:
ALPHA = 0.5   # soup weight on the incumbent; (1-ALPHA) on v34
soup = {}
n_avg = n_keep = 0
for k in sd_i:
    ti, tv = sd_i[k], sd_v[k]
    if torch.is_floating_point(ti):
        soup[k] = (ALPHA * ti.float() + (1.0 - ALPHA) * tv.float()).to(ti.dtype)
        n_avg += 1
    else:
        soup[k] = ti.clone()   # integer buffers (e.g. num_batches_tracked): keep incumbent
        n_keep += 1
print(f"averaged {n_avg} float tensors; kept {n_keep} integer buffers from incumbent")

# sanity: the soup must differ from BOTH parents (else averaging was a no-op)
import random
probe = [k for k in sd_i if torch.is_floating_point(sd_i[k]) and sd_i[k].numel() > 10][:5]
for k in probe:
    di = (soup[k].float() - sd_i[k].float()).abs().mean().item()
    dv = (soup[k].float() - sd_v[k].float()).abs().mean().item()
    par = (sd_i[k].float() - sd_v[k].float()).abs().mean().item()
    print(f"  {k[:48]:48} |soup-inc|={di:.4e} |soup-v34|={dv:.4e} |inc-v34|={par:.4e}")
assert all((soup[k].float()-sd_i[k].float()).abs().mean().item()>0 for k in probe), "soup == incumbent?!"


In [ ]:
# save souped checkpoint in the SAME layout the inference pipeline expects
out = Path("/kaggle/working/unet_transformer/split_0")
out.mkdir(parents=True, exist_ok=True)
torch.save(soup, out / "edge_predictor_best.pth")

# copy the (identical) config.json beside it
cfg = glob.glob("/kaggle/input/**/unet_transformer/split_0/config.json", recursive=True)
assert cfg, "config.json not found"
import shutil; shutil.copy(cfg[0], out / "config.json")

saved = out / "edge_predictor_best.pth"
sha = hashlib.sha256(saved.read_bytes()).hexdigest()
print(f"saved {saved}  ({saved.stat().st_size/1e6:.2f} MB)  sha {sha[:16]}")
print("config:", (out/"config.json").read_text())
# reload to confirm it is a valid checkpoint
_chk = torch.load(saved, map_location="cpu", weights_only=True)
assert set(_chk) == set(soup), "reload mismatch"
print("reload OK -", len(_chk), "tensors. Soup ready to package as a dataset.")
